# Módulo 1 — Práctica Día 1: sintaxis esencial y estructuras de datos

Ejercicios de la práctica del **Día 1** (ver [`README.md`](../README.md)).
Usamos mediciones cada hora de 5 estaciones automáticas de SENAMHI en Lima,
marzo de 2025 (carpeta `data/senamhi/`). No se descarga nada de internet.

## Cómo trabajar en este notebook

Cada ejercicio tiene tres partes, siempre en este orden:

1. **📘 Ejemplo resuelto.** Ejecútalo (Shift+Enter) y léelo línea por línea:
   es el modelo a imitar.
2. **✍️ Tu turno.** Un problema parecido, con otra estación u otro día. La
   celda ya corre sin errores, pero no hace nada útil hasta que completes
   las líneas marcadas con `✍️` (reemplaza el `None`, el `0`, el `"??"` o el
   `pass` que aparece ahí).
3. **✔️ Comprobar.** Ejecútala después de tu turno: dice `✅` si tu
   resultado es correcto, `❌` con una pista si no lo es, o `⏳` si todavía
   no completaste nada.

Antes de escribir, guarda una copia propia con *File → Save Notebook As…*
y el nombre `02_practica_dia1_mio.ipynb`, en esta misma carpeta (ver
[`antes_del_curso.md`](../../../antes_del_curso.md), sección 5).

## Antes de empezar — leer un archivo de SENAMHI

Cada archivo es texto: una línea por hora, con los valores separados por
`;`. Por ejemplo, la línea

```
2025;03;14;13:00;27.8;0.0;56;262;3.8
```

dice: año 2025, mes 03, día 14, a las 13:00 (hora de Perú), 27.8 °C, 0.0 mm
de lluvia, etc. La primera línea del archivo trae los nombres de las
columnas. Donde la estación no midió, dice **`S/D`** («sin dato»).

Con lo visto hoy (listas, diccionarios, bucles, funciones) ya podemos leerlo:

In [1]:
CARPETA = "../../../data/senamhi/"


def leer_senamhi_basico(archivo):
    """Lee un CSV de SENAMHI y devuelve una lista de filas; cada fila es un diccionario."""
    filas = []
    with open(CARPETA + archivo, encoding="utf-8") as f:
        encabezado = f.readline().strip().split(";")    # primera línea: nombres de columna
        for linea in f:                                   # el resto: una línea por hora
            valores = linea.strip().split(";")
            filas.append(dict(zip(encabezado, valores)))  # zip junta cada nombre con su valor
    return filas


def a_numero(texto):
    """Convierte el texto '27.8' en el número 27.8, y 'S/D' (sin dato) en None."""
    if texto == "S/D":
        return None
    return float(texto)


filas = leer_senamhi_basico("472AC278_2025_03.csv")   # Von Humboldt, La Molina
print("Número de filas (horas):", len(filas))
print("Primera fila:", filas[0])

Número de filas (horas): 743
Primera fila: {'Año': '2025', 'Mes': '03', 'Día': '01', 'Hora': '00:00', 'Temperatura (°C)': '22.3', 'Precipitación (mm)': '0.0', 'Humedad (%)': '87', 'Dir. Viento (°)': '250', 'Vel. Viento (m/s)': '0.1'}


## Ejercicio 1 — Estadísticas manuales sobre una lista (30–45 min)

Antes de usar NumPy (eso es el Día 2), calculamos a mano el promedio, el
máximo, el mínimo y cuántas horas superaron un umbral, usando solo listas,
bucles y condicionales.

### 📘 Ejemplo resuelto

Las 24 temperaturas del **14 de marzo de 2025** en la estación **Von
Humboldt** (La Molina). La hora es la local de Perú.

In [2]:
temperaturas_horarias = []
for fila in filas:
    if fila["Día"] == "14":
        temperaturas_horarias.append(a_numero(fila["Temperatura (°C)"]))

print("Von Humboldt, 14/03/2025 — temperaturas horarias (°C):")
print(temperaturas_horarias)

Von Humboldt, 14/03/2025 — temperaturas horarias (°C):
[21.7, 21.5, 21.3, 20.9, 20.6, 20.3, 20.3, 20.5, 21.7, 23.8, 26.1, 27.6, 27.9, 27.8, 28.3, 29.1, 26.8, 26.0, 24.5, 22.6, 22.4, 22.2, 21.9, 21.4]


In [3]:
# Todavía no vimos Matplotlib (eso es el Día 2) -- una forma simple de
# "visualizar" sin librerías extra es una barra hecha de caracteres.
print("Temperaturas horarias — barra de caracteres (1 '█' por °C)")
for hora, valor in enumerate(temperaturas_horarias):
    barra = "█" * max(int(valor), 0)
    print(f"h{hora:02d}  {valor:5.1f} °C | {barra}")

Temperaturas horarias — barra de caracteres (1 '█' por °C)
h00   21.7 °C | █████████████████████
h01   21.5 °C | █████████████████████
h02   21.3 °C | █████████████████████
h03   20.9 °C | ████████████████████
h04   20.6 °C | ████████████████████
h05   20.3 °C | ████████████████████
h06   20.3 °C | ████████████████████
h07   20.5 °C | ████████████████████
h08   21.7 °C | █████████████████████
h09   23.8 °C | ███████████████████████
h10   26.1 °C | ██████████████████████████
h11   27.6 °C | ███████████████████████████
h12   27.9 °C | ███████████████████████████
h13   27.8 °C | ███████████████████████████
h14   28.3 °C | ████████████████████████████
h15   29.1 °C | █████████████████████████████
h16   26.8 °C | ██████████████████████████
h17   26.0 °C | ██████████████████████████
h18   24.5 °C | ████████████████████████
h19   22.6 °C | ██████████████████████
h20   22.4 °C | ██████████████████████
h21   22.2 °C | ██████████████████████
h22   21.9 °C | █████████████████████
h23   21.4 °C | 

In [4]:
def resumen_temperaturas(valores, umbral=26.0):
    suma = 0.0
    maximo = valores[0]
    minimo = valores[0]
    horas_sobre_umbral = 0

    for v in valores:
        suma += v
        if v > maximo:
            maximo = v
        if v < minimo:
            minimo = v
        if v > umbral:
            horas_sobre_umbral += 1

    promedio = suma / len(valores)
    return {
        "promedio": round(promedio, 2),
        "maximo": maximo,
        "minimo": minimo,
        "horas_sobre_umbral": horas_sobre_umbral,
    }

resumen = resumen_temperaturas(temperaturas_horarias, umbral=26.0)
print(resumen)

{'promedio': 23.63, 'maximo': 29.1, 'minimo': 20.3, 'horas_sobre_umbral': 7}


### ✍️ Tu turno — Ejercicio 1

Haz lo mismo para la estación **Ñaña** (archivo `111290_2025_03.csv`) el
**20 de marzo**, con un umbral de **24 °C**.

Pistas:

- Lee el archivo con `leer_senamhi_basico(...)`, como en la primera celda.
- El día se compara como **texto**: `"20"`, con comillas.
- No reescribas `resumen_temperaturas`: ya existe y acepta otro `umbral`.

In [5]:
# ✍️ TU TURNO — completa las líneas marcadas con ✍️
filas_nana = []                       # ✍️ cambia [] por la lectura del archivo de Ñaña

temperaturas_nana = []
for fila in filas_nana:
    if fila["Día"] == "??":           # ✍️ ¿qué día?
        temperaturas_nana.append(None)  # ✍️ ¿qué valor agregas? (usa a_numero)

resumen_nana = None                   # ✍️ usa resumen_temperaturas(...) con umbral 24
print(resumen_nana)

None


In [6]:
# ✔️ COMPROBAR — ejecuta esta celda después de completar la anterior
esperado = {"promedio": 23.54, "maximo": 27.0, "minimo": 20.9, "horas_sobre_umbral": 8}
if resumen_nana is None:
    print("⏳ Todavía no completas la celda anterior.")
elif resumen_nana == esperado:
    print("✅ ¡Correcto!", resumen_nana)
else:
    print("❌ Aún no coincide. Obtuviste:", resumen_nana)
    print("   Revisa: ¿leíste el archivo de Ñaña?, ¿el día es \"20\" (con comillas)?,",
          "¿pasaste umbral=24?, ¿la lista tiene 24 valores?", len(temperaturas_nana))

⏳ Todavía no completas la celda anterior.


## Ejercicio 2 — Diccionarios anidados: estaciones meteorológicas (45 min)

### 📘 Ejemplo resuelto

Un diccionario con las 5 estaciones de SENAMHI: nombre, coordenadas, altura
y las 24 temperaturas del mismo día (14 de marzo). Después, tres funciones
que lo recorren.

In [7]:
# Código SENAMHI -> (nombre, latitud, longitud, altura en m)
# Copiado de data/senamhi/estaciones_lima.json
datos_estaciones = {
    "112265":   ("San Martín de Porres", -12.009, -77.084, 56),
    "112181":   ("Campo de Marte",       -12.071, -77.043, 117),
    "472AC278": ("Von Humboldt",         -12.082, -76.939, 247),
    "112278":   ("Ceres",                -12.029, -76.927, 339),
    "111290":   ("Ñaña",                 -11.987, -76.842, 553),
}

estaciones = {}
for codigo, (nombre, lat, lon, altura) in datos_estaciones.items():
    temperaturas = []
    for fila in leer_senamhi_basico(codigo + "_2025_03.csv"):
        if fila["Día"] == "14":
            temperaturas.append(a_numero(fila["Temperatura (°C)"]))
    estaciones[nombre] = {
        "lat": lat,
        "lon": lon,
        "altura": altura,
        "temperaturas": temperaturas,
    }

estaciones["Ceres"]

{'lat': -12.029,
 'lon': -76.927,
 'altura': 339,
 'temperaturas': [23.0,
  22.8,
  22.5,
  22.1,
  21.8,
  21.5,
  21.6,
  21.5,
  22.2,
  24.1,
  25.8,
  27.4,
  28.7,
  28.6,
  28.7,
  28.3,
  26.9,
  26.0,
  25.7,
  24.3,
  23.7,
  24.5,
  24.5,
  24.4]}

In [8]:
def estacion_mas_calida(estaciones):
    mejor_nombre = None
    mejor_promedio = float("-inf")
    for nombre, datos in estaciones.items():
        promedio = sum(datos["temperaturas"]) / len(datos["temperaturas"])
        if promedio > mejor_promedio:
            mejor_promedio = promedio
            mejor_nombre = nombre
    return mejor_nombre, round(mejor_promedio, 2)


def filtrar_por_altura(estaciones, altura_min, altura_max):
    return {
        nombre: datos
        for nombre, datos in estaciones.items()
        if altura_min <= datos["altura"] <= altura_max
    }


def resumen_por_estacion(estaciones):
    for nombre, datos in estaciones.items():
        promedio = sum(datos["temperaturas"]) / len(datos["temperaturas"])
        print(f"{nombre:20s} | {datos['altura']:3d} m | lat={datos['lat']:7.3f} lon={datos['lon']:7.3f} "
              f"| temp. promedio = {promedio:5.2f} °C")


nombre, promedio = estacion_mas_calida(estaciones)
print(f"Estación más cálida el 14/03: {nombre} ({promedio} °C)\n")

# En Lima todas están casi a la misma latitud: lo que cambia es la altura
lejos_del_mar = filtrar_por_altura(estaciones, altura_min=200, altura_max=600)
print("Estaciones entre 200 y 600 m de altura:", list(lejos_del_mar.keys()))
print()

resumen_por_estacion(estaciones)

Estación más cálida el 14/03: Ceres (24.61 °C)

Estaciones entre 200 y 600 m de altura: ['Von Humboldt', 'Ceres', 'Ñaña']

San Martín de Porres |  56 m | lat=-12.009 lon=-77.084 | temp. promedio = 22.65 °C
Campo de Marte       | 117 m | lat=-12.071 lon=-77.043 | temp. promedio = 22.31 °C
Von Humboldt         | 247 m | lat=-12.082 lon=-76.939 | temp. promedio = 23.63 °C
Ceres                | 339 m | lat=-12.029 lon=-76.927 | temp. promedio = 24.61 °C
Ñaña                 | 553 m | lat=-11.987 lon=-76.842 | temp. promedio = 23.68 °C


In [9]:
print("Temperatura promedio por estación — barra de caracteres, ordenado de mayor a menor")
promedios = {n: sum(d["temperaturas"]) / len(d["temperaturas"]) for n, d in estaciones.items()}
for n, p in sorted(promedios.items(), key=lambda kv: -kv[1]):
    barra = "█" * max(int(p), 0)
    print(f"{n:20s} {p:5.2f} °C | {barra}")

Temperatura promedio por estación — barra de caracteres, ordenado de mayor a menor
Ceres                24.61 °C | ████████████████████████
Ñaña                 23.68 °C | ███████████████████████
Von Humboldt         23.63 °C | ███████████████████████
San Martín de Porres 22.65 °C | ██████████████████████
Campo de Marte       22.31 °C | ██████████████████████


> ⚠️ **Ojo con los huecos.** Estas funciones usan `sum(...)`, que falla con
> `TypeError` si la lista tiene algún `None` (una hora `S/D`). El 14 de
> marzo no tiene huecos en ninguna estación; si cambias el día al 13, verás
> el error en Campo de Marte. En el Día 2 lo resolvemos con `NaN`.

### ✍️ Tu turno — Ejercicio 2

Escribe dos funciones, imitando a `estacion_mas_calida`:

- **(a)** `estacion_mas_fria(estaciones)`: devuelve `(nombre, promedio)` de
  la estación con el promedio **más bajo**, con el promedio redondeado a 2
  decimales.
- **(b)** `amplitud_termica(estaciones)`: devuelve un diccionario
  `{nombre: máxima − mínima}`, redondeado a 1 decimal. Pista: `max(lista)` y
  `min(lista)` funcionan sobre una lista de números.

In [10]:
# ✍️ TU TURNO — completa las líneas marcadas con ✍️
def estacion_mas_fria(estaciones):
    peor_nombre = None
    peor_promedio = float("inf")      # "infinito": cualquier promedio será menor
    for nombre, datos in estaciones.items():
        promedio = 0                  # ✍️ cambia 0 por el promedio de datos["temperaturas"]
        pass                          # ✍️ si promedio < peor_promedio, guarda nombre y promedio
    return peor_nombre, round(peor_promedio, 2)


def amplitud_termica(estaciones):
    resultado = {}
    for nombre, datos in estaciones.items():
        pass                          # ✍️ resultado[nombre] = máxima - mínima, redondeada a 1 decimal
    return resultado


print("(a)", estacion_mas_fria(estaciones))
print("(b)", amplitud_termica(estaciones))

(a) (None, inf)
(b) {}


In [11]:
# ✔️ COMPROBAR
fria = estacion_mas_fria(estaciones)
amplitud = amplitud_termica(estaciones)

if fria[0] is None:
    print("⏳ (a) Todavía no completas estacion_mas_fria.")
elif fria == ("Campo de Marte", 22.31):
    print("✅ (a) ¡Correcto!", fria)
else:
    print("❌ (a) Obtuviste", fria, "— ¿comparas con < (menor) y guardas el nombre y el promedio?")

esperado = {"San Martín de Porres": 6.6, "Campo de Marte": 6.1, "Von Humboldt": 8.8,
            "Ceres": 7.2, "Ñaña": 5.0}
if not amplitud:
    print("⏳ (b) Todavía no completas amplitud_termica.")
elif amplitud == esperado:
    print("✅ (b) ¡Correcto!", amplitud)
else:
    print("❌ (b) Obtuviste", amplitud, "— ¿redondeaste con round(..., 1)?")

⏳ (a) Todavía no completas estacion_mas_fria.
⏳ (b) Todavía no completas amplitud_termica.


## Ejercicio 2b — ¿Dónde faltan datos? (15 min)

### 📘 Ejemplo resuelto

Los archivos de las estaciones tienen huecos. Recorremos el mes completo de
cada estación y anotamos las horas cuya **temperatura** dice `S/D`.

In [12]:
for codigo, (nombre, lat, lon, altura) in datos_estaciones.items():
    filas_estacion = leer_senamhi_basico(codigo + "_2025_03.csv")
    faltantes = []
    for fila in filas_estacion:
        if a_numero(fila["Temperatura (°C)"]) is None:
            faltantes.append(fila["Día"] + "/03 " + fila["Hora"])
    print(f"{nombre:20s} {len(filas_estacion)} horas leídas, {len(faltantes)} con S/D: {faltantes}")

print()
print("Marzo tiene 31 días x 24 horas =", 31 * 24, "horas")

San Martín de Porres 737 horas leídas, 0 con S/D: []
Campo de Marte       744 horas leídas, 2 con S/D: ['11/03 22:00', '13/03 23:00']
Von Humboldt         743 horas leídas, 0 con S/D: []
Ceres                737 horas leídas, 0 con S/D: []
Ñaña                 744 horas leídas, 2 con S/D: ['09/03 06:00', '13/03 22:00']

Marzo tiene 31 días x 24 horas = 744 horas


Hay **dos tipos de hueco**:

- **`S/D`**: la fila existe, pero sin valor.
- **Filas que faltan**: algunas estaciones tienen menos de 744 horas. Esas
  horas no están en el archivo, y un bucle no las «ve».

Conviene buscar los dos tipos antes de calcular nada. Lo retomamos en el
Día 5.

### ✍️ Tu turno — Ejercicio 2b

Cuenta cuántas horas dicen `S/D` en la columna **`"Precipitación (mm)"`** de
cada estación y guárdalo en el diccionario `sd_lluvia` (`{nombre:
cantidad}`). ¿Qué estación no midió la lluvia? Lo necesitarás en el
Ejercicio 3.

In [13]:
# ✍️ TU TURNO — completa las líneas marcadas con ✍️
sd_lluvia = {}
for codigo, (nombre, lat, lon, altura) in datos_estaciones.items():
    contador = 0
    # ✍️ recorre las filas de leer_senamhi_basico(codigo + "_2025_03.csv")
    #    y suma 1 a contador cada vez que "Precipitación (mm)" diga "S/D"
    sd_lluvia[nombre] = contador

print(sd_lluvia)

{'San Martín de Porres': 0, 'Campo de Marte': 0, 'Von Humboldt': 0, 'Ceres': 0, 'Ñaña': 0}


In [14]:
# ✔️ COMPROBAR
esperado = {"San Martín de Porres": 0, "Campo de Marte": 3, "Von Humboldt": 0,
            "Ceres": 737, "Ñaña": 3}
if sum(sd_lluvia.values()) == 0:
    print("⏳ Todavía no completas la celda anterior (todo da 0).")
elif sd_lluvia == esperado:
    print("✅ ¡Correcto! Ceres no midió lluvia ninguna hora:", sd_lluvia["Ceres"], "horas con S/D.")
else:
    print("❌ Obtuviste", sd_lluvia, "— ¿comparas con el texto \"S/D\" exacto?")

⏳ Todavía no completas la celda anterior (todo da 0).


## Ejercicio 3 — Clasificación de precipitación (45 min)

### 📘 Ejemplo resuelto

Función reutilizable `clasificar_precipitacion(valor)` aplicada a la lluvia
de cada día de marzo en la estación **Ñaña**. El archivo trae la lluvia de
cada hora: primero hay que sumar las horas de cada día.

Cuidado con los huecos: si un día no tiene **ninguna** medición, su total
no es «0 mm» (no llovió), sino «sin dato» (no sabemos). Por eso cada día
empieza en `None` y pasa a `0.0` recién con la primera medición.

In [15]:
def clasificar_precipitacion(valor_mm):
    """Clasifica un valor diario de precipitación (mm) en una categoría."""
    if valor_mm is None:
        return "sin dato"
    elif valor_mm <= 0.1:
        return "sin lluvia"
    elif valor_mm <= 2.0:
        return "llovizna"
    elif valor_mm <= 15.0:
        return "moderada"
    else:
        return "fuerte"


def lluvia_diaria(archivo):
    """Suma la lluvia de cada hora: {"01": 0.0, "02": 0.4, ...}; None si el día no tiene ninguna medición."""
    totales = {}
    for fila in leer_senamhi_basico(archivo):
        dia = fila["Día"]
        valor = a_numero(fila["Precipitación (mm)"])
        if dia not in totales:
            totales[dia] = None            # todavía no hay ninguna medición
        if valor is not None:
            if totales[dia] is None:
                totales[dia] = 0.0         # primera medición del día
            totales[dia] += valor
    return totales


precipitacion_diaria = lluvia_diaria("111290_2025_03.csv")   # Ñaña

for dia, valor in precipitacion_diaria.items():
    if valor is None:
        print(f"{dia}/03:  S/D    -> {clasificar_precipitacion(valor)}")
    else:
        valor = round(valor, 1)        # recuerda 0.1 + 0.2 de la teoría: redondeamos
        print(f"{dia}/03: {valor:4.1f} mm -> {clasificar_precipitacion(valor)}")

01/03:  0.0 mm -> sin lluvia
02/03:  0.0 mm -> sin lluvia
03/03:  0.0 mm -> sin lluvia
04/03:  0.0 mm -> sin lluvia
05/03:  0.0 mm -> sin lluvia
06/03:  0.0 mm -> sin lluvia
07/03:  0.0 mm -> sin lluvia
08/03:  0.0 mm -> sin lluvia
09/03:  0.0 mm -> sin lluvia
10/03:  0.0 mm -> sin lluvia
11/03:  0.0 mm -> sin lluvia
12/03:  0.0 mm -> sin lluvia
13/03:  0.0 mm -> sin lluvia
14/03:  0.0 mm -> sin lluvia
15/03:  0.4 mm -> llovizna
16/03:  0.0 mm -> sin lluvia
17/03:  0.0 mm -> sin lluvia
18/03:  0.1 mm -> sin lluvia
19/03:  0.0 mm -> sin lluvia
20/03:  0.0 mm -> sin lluvia
21/03:  0.0 mm -> sin lluvia
22/03:  0.0 mm -> sin lluvia
23/03:  0.0 mm -> sin lluvia
24/03:  0.0 mm -> sin lluvia
25/03:  0.0 mm -> sin lluvia
26/03:  0.0 mm -> sin lluvia
27/03:  1.1 mm -> llovizna
28/03:  0.0 mm -> sin lluvia
29/03:  0.0 mm -> sin lluvia
30/03:  0.0 mm -> sin lluvia
31/03:  0.0 mm -> sin lluvia


In [16]:
print("Precipitación diaria en Ñaña — barra de caracteres (1 '█' por cada 0.1 mm)")
for dia, valor in precipitacion_diaria.items():
    categoria = clasificar_precipitacion(valor)
    if valor is None:
        print(f"{dia}/03:  S/D    | {'':12s} {categoria}")
        continue
    barra = "█" * int(round(valor * 10))
    print(f"{dia}/03: {valor:4.1f} mm | {barra:12s} {categoria}")

Precipitación diaria en Ñaña — barra de caracteres (1 '█' por cada 0.1 mm)
01/03:  0.0 mm |              sin lluvia
02/03:  0.0 mm |              sin lluvia
03/03:  0.0 mm |              sin lluvia
04/03:  0.0 mm |              sin lluvia
05/03:  0.0 mm |              sin lluvia
06/03:  0.0 mm |              sin lluvia
07/03:  0.0 mm |              sin lluvia
08/03:  0.0 mm |              sin lluvia
09/03:  0.0 mm |              sin lluvia
10/03:  0.0 mm |              sin lluvia
11/03:  0.0 mm |              sin lluvia
12/03:  0.0 mm |              sin lluvia
13/03:  0.0 mm |              sin lluvia
14/03:  0.0 mm |              sin lluvia
15/03:  0.4 mm | ████         llovizna
16/03:  0.0 mm |              sin lluvia
17/03:  0.0 mm |              sin lluvia
18/03:  0.1 mm | █            sin lluvia
19/03:  0.0 mm |              sin lluvia
20/03:  0.0 mm |              sin lluvia
21/03:  0.0 mm |              sin lluvia
22/03:  0.0 mm |              sin lluvia
23/03:  0.0 mm |         

En marzo de 2025 casi no llovió en Lima, como es normal en la costa
peruana. Ñaña, la estación más alta y alejada del mar, tuvo solo 2 días con
llovizna.

La categoría `"sin dato"` también aparece: la estación Ceres no registró
lluvia ningún día de marzo (todas sus horas dicen `S/D`, como viste en el
Ejercicio 2b). Gracias a `lluvia_diaria`, esos días salen como «sin dato» y
no como «sin lluvia»:

In [17]:
lluvia_ceres = lluvia_diaria("112278_2025_03.csv")
for dia in ["01", "02", "03"]:
    print(f"Ceres {dia}/03: {lluvia_ceres[dia]} -> {clasificar_precipitacion(lluvia_ceres[dia])}")

Ceres 01/03: None -> sin dato
Ceres 02/03: None -> sin dato
Ceres 03/03: None -> sin dato


### ✍️ Tu turno — Ejercicio 3

Escribe `clasificar_temperatura(valor_c)` para la temperatura **máxima** de
un día:

| Máxima del día | Categoría |
|---|---|
| sin dato (`None`) | `"sin dato"` |
| hasta 25 °C (incluido) | `"templado"` |
| más de 25 y hasta 28 °C (incluido) | `"cálido"` |
| más de 28 °C | `"muy cálido"` |

Imita la estructura `if / elif / else` de `clasificar_precipitacion`.

In [18]:
# ✍️ TU TURNO — completa la función
def clasificar_temperatura(valor_c):
    """Clasifica la temperatura máxima diaria (°C)."""
    return "??"                       # ✍️ reemplaza esta línea por tu if / elif / else

In [19]:
# ✔️ COMPROBAR — prueba la función con valores cuya respuesta conocemos
casos = {None: "sin dato", 24.6: "templado", 25.0: "templado", 27.4: "cálido",
         28.0: "cálido", 30.5: "muy cálido"}
errores = {v: clasificar_temperatura(v) for v, c in casos.items() if clasificar_temperatura(v) != c}
if clasificar_temperatura(25.0) == "??":
    print("⏳ Todavía no completas clasificar_temperatura.")
elif not errores:
    print("✅ ¡Correcto! La función clasifica bien los", len(casos), "casos de prueba.")
else:
    print("❌ Estos valores salen mal (valor: lo que devolvió tu función):", errores)
    print("   Revisa los límites: 25.0 es \"templado\" y 28.0 es \"cálido\" (<=, no <).")

⏳ Todavía no completas clasificar_temperatura.


Con la función lista, la aplicamos a la máxima de cada día de **Campo de
Marte** (las horas `S/D` se saltan con `continue`):

In [20]:
maximas = {}
for fila in leer_senamhi_basico("112181_2025_03.csv"):     # Campo de Marte
    valor = a_numero(fila["Temperatura (°C)"])
    if valor is None:
        continue                      # saltamos las horas S/D
    dia = fila["Día"]
    if dia not in maximas or valor > maximas[dia]:
        maximas[dia] = valor

for dia, tmax in maximas.items():
    print(f"{dia}/03: {tmax:4.1f} °C -> {clasificar_temperatura(tmax)}")

01/03: 29.0 °C -> ??
02/03: 30.5 °C -> ??
03/03: 29.7 °C -> ??
04/03: 27.0 °C -> ??
05/03: 27.0 °C -> ??
06/03: 27.4 °C -> ??
07/03: 26.9 °C -> ??
08/03: 27.4 °C -> ??
09/03: 28.4 °C -> ??
10/03: 27.5 °C -> ??
11/03: 25.6 °C -> ??
12/03: 25.4 °C -> ??
13/03: 25.5 °C -> ??
14/03: 26.1 °C -> ??
15/03: 27.3 °C -> ??
16/03: 26.7 °C -> ??
17/03: 26.6 °C -> ??
18/03: 24.7 °C -> ??
19/03: 24.6 °C -> ??
20/03: 28.2 °C -> ??
21/03: 27.0 °C -> ??
22/03: 25.9 °C -> ??
23/03: 28.0 °C -> ??
24/03: 25.3 °C -> ??
25/03: 26.6 °C -> ??
26/03: 27.4 °C -> ??
27/03: 27.6 °C -> ??
28/03: 29.4 °C -> ??
29/03: 28.9 °C -> ??
30/03: 28.1 °C -> ??
31/03: 29.0 °C -> ??


## Ejercicio 4 — De notebook a script `.py` (30 min)

### 📘 Ejemplo resuelto

Un **script** es un archivo de texto con código Python que se ejecuta desde
la terminal, sin Jupyter. La celda siguiente empieza con `%%writefile`:
guarda el resto de la celda como el archivo `clasificar_precipitacion.py`
en esta carpeta. Es lo mismo que crearlo en el editor de JupyterLab
(*File → New → Python File*) y pegar el texto.

Fíjate en `Path(__file__).parent`: es la carpeta donde está guardado el
script. Gracias a eso, el script encuentra los datos aunque lo ejecutes
desde otra carpeta de la terminal.

In [21]:
%%writefile clasificar_precipitacion.py
"""Clasificacion de la lluvia diaria (mm) de una estacion de SENAMHI."""
import sys
from pathlib import Path

# Carpeta de los datos, relativa a la carpeta de ESTE archivo (no a la terminal)
CARPETA_DATOS = Path(__file__).resolve().parent / "../../../data/senamhi"


def clasificar_precipitacion(valor_mm):
    if valor_mm is None:
        return "sin dato"
    elif valor_mm <= 0.1:
        return "sin lluvia"
    elif valor_mm <= 2.0:
        return "llovizna"
    elif valor_mm <= 15.0:
        return "moderada"
    else:
        return "fuerte"


def lluvia_diaria(ruta):
    """Total de cada dia; None si el dia no tiene ninguna medicion."""
    totales = {}
    with open(ruta, encoding="utf-8") as f:
        encabezado = f.readline().strip().split(";")
        for linea in f:
            fila = dict(zip(encabezado, linea.strip().split(";")))
            dia = fila["Día"]
            if dia not in totales:
                totales[dia] = None
            if fila["Precipitación (mm)"] != "S/D":
                if totales[dia] is None:
                    totales[dia] = 0.0
                totales[dia] += float(fila["Precipitación (mm)"])
    return totales


def main():
    # Sin argumento usa Ñaña; se puede pasar otro archivo:
    #   python3 clasificar_precipitacion.py ../../../data/senamhi/112181_2025_03.csv
    if len(sys.argv) > 1:
        ruta = sys.argv[1]
    else:
        ruta = CARPETA_DATOS / "111290_2025_03.csv"
    for dia, valor in lluvia_diaria(ruta).items():
        if valor is None:
            print(f"Dia {dia}/03:  S/D    -> {clasificar_precipitacion(valor)}")
        else:
            valor = round(valor, 1)
            print(f"Dia {dia}/03: {valor:4.1f} mm -> {clasificar_precipitacion(valor)}")


if __name__ == "__main__":
    main()

Overwriting clasificar_precipitacion.py


Ahora lo ejecutamos **como script**. En la terminal (con el entorno activo y
desde esta carpeta) escribirías `python3 clasificar_precipitacion.py`; en
Jupyter, el signo `!` al inicio de una línea hace lo mismo. Primero con Ñaña
(por defecto) y luego con Ceres, pasando su archivo como argumento:

In [22]:
!python3 clasificar_precipitacion.py | head -5
!python3 clasificar_precipitacion.py ../../../data/senamhi/112278_2025_03.csv | head -3

Dia 01/03:  0.0 mm -> sin lluvia
Dia 02/03:  0.0 mm -> sin lluvia
Dia 03/03:  0.0 mm -> sin lluvia
Dia 04/03:  0.0 mm -> sin lluvia
Dia 05/03:  0.0 mm -> sin lluvia


Dia 01/03:  S/D    -> sin dato
Dia 02/03:  S/D    -> sin dato
Dia 03/03:  S/D    -> sin dato


### ✍️ Tu turno — Ejercicio 4

Crea **en el editor** (*File → New → Python File*) un script llamado
`temperatura_maxima_mio.py`, en esta misma carpeta, que imprima la
temperatura máxima de cada día del archivo que recibe como argumento. Usa
el script de arriba como modelo: copia la lectura del archivo, cambia la
suma de la lluvia por la búsqueda del máximo (como en el Ejercicio 3) y
salta las horas `S/D`. La salida de cada línea debe verse así:

```
Dia 01/03: 29.0 °C
```

Pruébalo en la terminal con
`python3 temperatura_maxima_mio.py ../../../data/senamhi/112181_2025_03.csv`
y después ejecuta la comprobación.

In [23]:
# ✔️ COMPROBAR — ejecuta tu script con Campo de Marte y revisa dos días conocidos
import subprocess
import sys
from pathlib import Path

if not Path("temperatura_maxima_mio.py").exists():
    print("⏳ Todavía no existe temperatura_maxima_mio.py en esta carpeta.")
else:
    r = subprocess.run([sys.executable, "temperatura_maxima_mio.py",
                        "../../../data/senamhi/112181_2025_03.csv"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("❌ El script terminó con un error:\n", r.stderr[-800:])
    elif "Dia 02/03: 30.5" in r.stdout and "Dia 19/03: 24.6" in r.stdout:
        print("✅ ¡Correcto! Primeras líneas de tu script:")
        print("\n".join(r.stdout.splitlines()[:3]))
    else:
        print("❌ El script corre, pero no encuentro «Dia 02/03: 30.5» ni «Dia 19/03: 24.6».")
        print("   Tu salida empieza así:\n", "\n".join(r.stdout.splitlines()[:3]))

⏳ Todavía no existe temperatura_maxima_mio.py en esta carpeta.


## Cierre

Entregable del día, en esta carpeta:

- tu copia `02_practica_dia1_mio.ipynb`, con todos los ✔️ en `✅`;
- `clasificar_precipitacion.py` (lo generó el ejemplo) y tu
  `temperatura_maxima_mio.py`.